# B1.7 · Feasibility filtering and reachability

**Function B — Application Security with an AI SDLC → The AI SDLC: an Agentic AppSec Pipeline**  ·  *AI for Security*

Builds on **[B1.6 · Deduplication and contextual verification](https://spbreed.github.io/cyber-commons/lessons/B1.6.html)**.

| | |
|---|---|
| Open-source tooling | CodeQL, tree-sitter |
| Open-weight models | GLM-4.6 |
| Frontier models | Claude Sonnet 5 |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off — and where a lesson involves a model, the same code calls an open-weight endpoint or a frontier API when you configure one.

## 1 · The hook

The finding is real. The code is dead. Reachability is the difference between a queue an engineer works and a queue an engineer learns to ignore, and it is the single largest false-positive killer in the pipeline.

## 2 · The framework

```
   is there a path from untrusted input to this line?

   HTTP handler --> parse() --> validate() --> build_query() --> DB
                                                    ^
                                              the finding

   reachable   -> a finding
   unreachable -> a note

   the largest single false-positive killer in the pipeline
```

**Stage 10 — Feasibility filtering.** The last stage of Phase 3, and the one
that decides whether anyone gets paged.

A verified finding is a real bug in the code. It is not necessarily a real risk,
because the code may be unreachable: dead code, a test fixture, an internal
function no external caller can drive, a branch behind a feature flag that has
been off for two years.

Triaging an unreachable finding costs exactly as much as triaging one on the
login path, and there are usually far more of them. So this stage partitions
findings into three buckets — and the third bucket is the honest one:

- **reachable** — a path exists from an untrusted entry point to the sink,
- **unreachable** — no path exists,
- **unknown** — the analysis cannot decide, usually because of dynamic dispatch,
  reflection, or a framework that wires callers at runtime.

Reporting `unknown` as `unreachable` is how a pipeline quietly drops real bugs.

> **Where you are in the pipeline.**
>
> ```
> [Ingestion & Mapping] ──> [Threat Modelling] ──> [Discovery]
>          └─ stages 1-4         └─ stages 5-6        └─ stages 7-10
>                    ──> [Dynamic Validation] ──> [Reporting]
>                              └─ stages 11-14        └─ stage 15
> ```

## 3 · Stage 10 — build the call graph from entry points

In [ ]:
import ast
from collections import defaultdict

SOURCE = '''
import handlers_registry

def http_get_report(request):
    """ENTRY: GET /reports"""
    return load_report(request.args["id"])

def http_health(request):
    """ENTRY: GET /health"""
    return "ok"

def load_report(report_id):
    return DB.execute("SELECT * FROM reports WHERE id=" + report_id)

def legacy_export(report_id):
    # nothing calls this any more; kept for a migration that finished in 2023
    return DB.execute("SELECT * FROM reports WHERE id=" + report_id)

def debug_dump(name):
    return open("/tmp/" + name).read()

def dispatch(request):
    """ENTRY: dynamic dispatch — the framework resolves the handler at runtime"""
    handler = handlers_registry.lookup(request.path)
    return handler(request)
'''

tree = ast.parse(SOURCE)
FUNCS = {fn.name: fn for fn in ast.walk(tree) if isinstance(fn, ast.FunctionDef)}

def calls_in(fn):
    return {(c.func.id if isinstance(c.func, ast.Name) else getattr(c.func, "attr", ""))
            for c in ast.walk(fn) if isinstance(c, ast.Call)} - {""}

GRAPH = {name: sorted(calls_in(fn) & set(FUNCS)) for name, fn in FUNCS.items()}
ENTRY = [n for n, fn in FUNCS.items() if (ast.get_docstring(fn) or "").startswith("ENTRY")]
DYNAMIC = [n for n, fn in FUNCS.items()
           if "dynamic dispatch" in (ast.get_docstring(fn) or "")]

print("call graph:")
for n, cs in GRAPH.items(): print(f"   {n:18s}→ {cs or '—'}")
print(f"\nentry points: {ENTRY}")
print(f"dynamic dispatch present in: {DYNAMIC}")

In [ ]:
SINKS = {"load_report": ("CWE-89", "DB.execute"),
         "legacy_export": ("CWE-89", "DB.execute"),
         "debug_dump":   ("CWE-22", "open")}

def reachable_from(entry, graph):
    seen, stack = set(), [entry]
    while stack:
        n = stack.pop()
        for m in graph.get(n, []):
            if m not in seen: seen.add(m); stack.append(m)
    return seen

REACHED = set()
for e in ENTRY: REACHED |= reachable_from(e, GRAPH) | {e}

def feasibility(unit):
    if unit in REACHED:
        return "reachable", f"path exists from {[e for e in ENTRY if unit in reachable_from(e, GRAPH) | {e}]}"
    if DYNAMIC:
        return "unknown", (f"no static path, but {DYNAMIC[0]}() resolves handlers at "
                           f"runtime — cannot prove unreachable")
    return "unreachable", "no path from any entry point"

print(f"{'finding':16s}{'cwe':9s}{'verdict':13s}why")
print("-" * 92)
buckets = defaultdict(list)
for unit, (cwe, sink) in SINKS.items():
    verdict, why = feasibility(unit)
    buckets[verdict].append(unit)
    print(f"{unit:16s}{cwe:9s}{verdict:13s}{why[:52]}")
print(f"\n{ {k: v for k, v in buckets.items()} }")

## 4 · Where it breaks — collapsing `unknown` into `unreachable`

The tempting simplification. It makes the queue shorter and it is how real bugs get dropped, because dynamic dispatch is exactly where framework-wired handlers live.

In [ ]:
def naive_filter(sinks, reached):
    """Two buckets. Anything not statically reached is discarded."""
    return {u: ("reachable" if u in reached else "unreachable") for u in sinks}

naive = naive_filter(SINKS, REACHED)
print(f"{'finding':16s}{'3-bucket':13s}{'2-bucket (naive)':18s}")
print("-" * 52)
for u in SINKS:
    v, _ = feasibility(u)
    print(f"{u:16s}{v:13s}{naive[u]:18s}"
          f"{'   ← DROPPED' if v == 'unknown' and naive[u] == 'unreachable' else ''}")

dropped = [u for u in SINKS if feasibility(u)[0] == "unknown"
           and naive[u] == "unreachable"]
print(f"\nfindings silently dropped by two-bucket filtering: {dropped}")
print("legacy_export is reachable through the runtime handler registry in this")
print("application. Static analysis cannot see that, and 'unreachable' is a lie.")
assert dropped

## 5 · The control — route each bucket to a different place

In [ ]:
ROUTING = {
 "reachable":   ("page / block the merge", "confirmed exploit path — goes to Phase 4"),
 "unknown":     ("queue for dynamic validation", "Phase 4 decides it empirically"),
 "unreachable": ("record, do not page", "revisit only if an entry point is added"),
}
for bucket, (action, why) in ROUTING.items():
    items = buckets.get(bucket, [])
    print(f"{bucket:13s}{len(items):>2} finding(s) → {action:28s}{why}")
    for i in items: print(f"{'':15s}{i}")

def queue_load(buckets, routing):
    paged = len(buckets.get("reachable", []))
    validated = len(buckets.get("unknown", []))
    silent = len(buckets.get("unreachable", []))
    return {"pages_a_human": paged, "sent_to_phase_4": paged + validated,
            "recorded_only": silent,
            "human_load_reduction": round(1 - paged / max(sum(map(len, buckets.values())), 1), 2)}

print(f"\n{queue_load(buckets, ROUTING)}")
print("\nThe unknown bucket is not a failure of the analysis. It is the handover")
print("to Phase 4, which answers reachability by running the thing.")

## 6 · Phase 3 as a skill — and the counts that police it

Stages 7 to 10 only ever *shrink* the list. That is a property worth enforcing rather than trusting, so the skill's contract carries a `counts` object and the rule that it must never increase.

A pipeline whose `verified` count exceeds its `deduped` count has invented findings somewhere after the audit stage — and that is far easier to do by accident than it sounds, because a verification step that expands one finding per code path looks perfectly reasonable from the inside.

In [ ]:
import json, re

def parse_skill(md):
    """Split a SKILL.md into (frontmatter dict, body).

    Frontmatter is a small, fixed subset of YAML: `key: value`, plus folded
    scalars (`description: >-`) whose continuation lines are indented. That is
    all a skill needs, and parsing it directly means no dependency.
    """
    if not md.startswith("---"):
        raise ValueError("a SKILL.md must open with a frontmatter block")
    _, front, body = md.split("---", 2)
    meta, key = {}, None
    for line in front.strip().splitlines():
        if not line.strip():
            continue
        if not line[0].isspace() and ":" in line:
            key, val = line.split(":", 1)
            key, val = key.strip(), val.strip()
            # `>-` and `|` open a folded block; the value is on the next lines
            meta[key] = "" if val in (">-", ">", "|", "|-") else val
        elif key is not None:
            meta[key] = (meta[key] + " " + line.strip()).strip()
    if "allowed-tools" in meta:
        meta["allowed-tools"] = [t.strip() for t in meta["allowed-tools"].split(",")
                                 if t.strip()]
    for required in ("name", "description"):
        if not meta.get(required):
            raise ValueError(f"skill is missing a {required!r}")
    return meta, body.strip()

_WORD = re.compile(r"[a-z][a-z-]{3,}")

def route(task, skills):
    """Pick the skill whose description best matches a task. Deterministic.

    The description is not documentation — it is the routing key. An agent
    decides whether to load a skill by reading it, so a vague description means
    the skill never fires when it should, and two overlapping descriptions mean
    the wrong one fires.

    Returns (pick, scores, margin). A margin of 0 means the top two scored the
    same and the "winner" is just whichever sorted first — an arbitrary answer
    wearing a confident face. Callers should refuse to auto-route on margin 0
    rather than pretend the tiebreak meant something.
    """
    want = set(_WORD.findall(task.lower()))
    def score(meta):
        return len(want & set(_WORD.findall(meta["description"].lower())))
    scores = {n: score(skills[n]) for n in sorted(skills)}
    # sort names first, then by score: ties must break identically on every
    # machine or the same task routes differently on two runs
    ranked = sorted(sorted(skills), key=lambda n: -scores[n])
    top = scores[ranked[0]]
    margin = top - (scores[ranked[1]] if len(ranked) > 1 else 0)
    return ranked[0], scores, margin

def contract_of(body):
    """The JSON block under '## Output contract' — the skill's machine promise."""
    # non-greedy across any prose between the heading and the fence
    m = re.search(r"## Output contract\b.*?```json\n(.*?)```", body, re.S)
    if not m:
        raise ValueError("skill declares no output contract")
    return json.loads(m.group(1))

def check(instance, contract, path="$"):
    """Structural conformance of an instance against a contract template.

    Returns the list of problems. An empty list means the shape is right — and
    that is *all* it means. Conformance is not accuracy: an empty findings list
    conforms perfectly and tells you nothing.
    """
    problems = []
    if isinstance(contract, dict):
        if not isinstance(instance, dict):
            return [f"{path}: expected an object, got {type(instance).__name__}"]
        for k, v in sorted(contract.items()):
            if k not in instance:
                problems.append(f"{path}.{k}: missing")
            else:
                problems += check(instance[k], v, f"{path}.{k}")
    elif isinstance(contract, list):
        if not isinstance(instance, list):
            return [f"{path}: expected a list, got {type(instance).__name__}"]
        for i, item in enumerate(instance):          # every element, same template
            problems += check(item, contract[0], f"{path}[{i}]")
    elif isinstance(contract, str) and "|" in contract:
        if instance not in contract.split("|"):
            problems.append(f"{path}: {instance!r} is not one of {contract}")
    elif isinstance(contract, bool):                  # before the numeric case:
        if not isinstance(instance, bool):            # bool is a subclass of int
            problems.append(f"{path}: expected bool, got {type(instance).__name__}")
    elif isinstance(contract, (int, float)):
        # JSON has one number type. A contract written `0` must accept 0.4, or
        # every cost and rate in the pipeline has to be rounded to satisfy a
        # checker rather than to be correct.
        if isinstance(instance, bool) or not isinstance(instance, (int, float)):
            problems.append(f"{path}: expected a number, got {type(instance).__name__}")
    elif not isinstance(instance, type(contract)):
        problems.append(f"{path}: expected {type(contract).__name__}, "
                        f"got {type(instance).__name__}")
    return problems

In [ ]:
# skills/appsec/appsec-vuln-audit/SKILL.md — embedded verbatim from the repository.
# This is the file itself, not a paraphrase of it.
SKILL_MD = r"""---
name: appsec-vuln-audit
description: >-
  Audit code for vulnerabilities against a threat model, then deduplicate,
  verify in context, and filter to what is actually reachable. Use when asked
  to review code for security bugs, run or interpret SAST, check whether a
  finding is a false positive, or reduce a noisy findings list to the ones
  worth a human's time.
allowed-tools: Read, Grep, Glob, Bash
---

# AppSec pipeline · Phase 3 — Analysis and filtering

Covers **stages 7–10**. This is where findings are produced — and, more
importantly, where most of them are thrown away.

The hard problem in application security is not finding candidate defects. It
is that a scanner emits hundreds and a human can act on ten. Every stage after
7 exists to shrink the list without losing the true positives.

## When to use this

When you have a threat model and a budget, or when handed a raw findings file
that nobody trusts. Stages 8–10 work on any findings list, including one from a
third-party scanner.

## Inputs

| Input | Required | Notes |
|---|---|---|
| `threat_model` + `plan.selected` | preferred | from appsec-threat-model |
| Source worktree | yes | verification needs the code, not just the finding |
| Existing findings | optional | run stages 8–10 alone to clean a noisy list |

## Procedure

**Stage 7 — Vulnerability auditing.** For each selected threat, examine the
path from entry to sink and decide whether the weakness is actually present.
Record for each finding: `cwe`, `file`, `line`, `unit`, the **evidence** (the
specific expression that is unsafe), and the **sanitiser** you looked for and
did not find. A finding that cannot name what was missing is a guess.

Three generations of analysis, and they are complementary, not competing:
grep-class pattern matching (fast, no dataflow), taint analysis (dataflow, no
semantics), and model-assisted review (semantics, no guarantees). Use the
cheapest one that can answer the question, and never let the third overrule the
second on a question of reachability — the model does not execute the program.

**Stage 8 — Deduplication.** The same defect appears many times: once per
scanner, once per path, once per call site. Collapse on the **defect identity**
— `(cwe, file, unit, sink_expression)` — not on the message text. Keep the
count: `occurrences` is signal about how exposed the defect is.

Match paths by parent directory plus filename tail. Deduplicating on a bare
basename silently merges two different files and loses a real finding.

**Stage 9 — Contextual verification.** For each surviving finding, look at the
surrounding code for the thing that makes it not-a-bug: a validator upstream, a
framework escaping the parameter, a type that cannot hold the payload, a caller
that only ever passes a constant. Record the verdict and the reason:
`confirmed`, `mitigated_by <what>`, or `needs_human`.

`needs_human` is a legitimate verdict and must stay available. A pipeline that
must decide will decide wrongly under uncertainty.

**Stage 10 — Feasibility filtering.** Drop what an attacker cannot actually
reach: code behind a feature flag that is off, an admin-only path in a service
with no admin, a sink whose input is fully constant. Record *why* each drop was
made, because the next scan will rediscover it and the reason is what stops
that work being repeated.

## Output contract

```json
{
  "findings": [
    {"id": "str", "cwe": "CWE-89", "file": "str", "line": 0, "unit": "str",
     "evidence": "str", "missing_control": "str",
     "occurrences": 1, "verdict": "confirmed|mitigated|needs_human",
     "verdict_reason": "str", "feasible": true, "confidence": 0.0}
  ],
  "dropped": [{"id": "str", "stage": 8, "why": "str"}],
  "counts": {"raw": 0, "deduped": 0, "verified": 0, "feasible": 0}
}
```

`counts` must be monotonically non-increasing across the four stages. If it is
not, the pipeline invented findings after the audit stage — stop and report.

## Failure modes

- **Confusing conformance with accuracy.** Output that matches this schema
  perfectly can still be entirely wrong. Schema validity is close to free;
  correctness is the expensive part. Never report conformance as a quality
  metric.
- **Dropping silently.** Every drop needs a stage and a reason.
- **Letting a model overrule dataflow on reachability.** It may propose a path;
  it may not confirm one.
- **Suppressing `needs_human` to look decisive.** Uncertainty that is hidden
  becomes someone's incident.

## Handoff

Feasible, confirmed findings go to **appsec-exploit-validate** for proof.
Everything else goes to **appsec-triage-report** with its verdict intact.
"""

meta, body = parse_skill(SKILL_MD)
print(f"loaded skill: {meta['name']}")
print(f"  tools it may use: {', '.join(meta.get('allowed-tools', [])) or '—'}")
print(f"  routing description: {len(meta['description'].split())} words")
print(f"  procedure: {len(body.splitlines())} lines")

In [ ]:
contract = contract_of(body)

FILE_OF = {"load_report": "src/data/reports.py", "legacy_export": "src/data/legacy.py",
           "debug_dump": "src/util/debug.py"}
MISSING = {"CWE-89": "parameterised query", "CWE-22": "path normalisation"}

findings = []
for unit, (cwe, sink) in sorted(SINKS.items()):
    verdict, why = feasibility(unit)
    findings.append({
        "id": f"F-{unit}", "cwe": cwe, "file": FILE_OF[unit], "line": 1,
        "unit": unit, "evidence": f"{sink} reached with caller-supplied input",
        "missing_control": MISSING[cwe], "occurrences": 1,
        # a finding we cannot prove reachable is not "confirmed" - it is the
        # one honest use of needs_human in the whole pipeline
        "verdict": "confirmed" if verdict == "reachable" else "needs_human",
        "verdict_reason": why,
        "feasible": verdict == "reachable",
        "confidence": 0.9 if verdict == "reachable" else 0.4})

audit = {
 "findings": findings,
 "dropped": [{"id": f"F-{u}", "stage": 10, "why": "no path from any entry point"}
             for u in sorted(buckets.get("unreachable", []))],
 # three analysers each reported every defect, so the raw count is 3x the
 # number of real defects. That is the normal case, not a bad day.
 "counts": {"raw": len(SINKS) * 3, "deduped": len(SINKS),
            "verified": len(findings),
            "feasible": sum(1 for f in findings if f["feasible"])},
}

problems = check(audit, contract)
print(f"conformance: {len(problems)} problem(s)")
for p in problems: print("   ", p)
assert not problems, problems

c = audit["counts"]
seq = [c["raw"], c["deduped"], c["verified"], c["feasible"]]
print(f"\ncounts raw->deduped->verified->feasible : {seq}")
print(f"monotonically non-increasing            : {all(x >= y for x, y in zip(seq, seq[1:]))}")
assert all(x >= y for x, y in zip(seq, seq[1:])), seq

## 7 · Where it breaks — deduplicating on the wrong key

The skill says to collapse on the **defect identity**, `(cwe, file, unit, sink_expression)`, and never on the message text. Here is why that sentence is in the procedure.

In [ ]:
# The same three defects, as three analysers actually report them.
ANALYSER_WORDING = {
 "grep rules":  "possible {cwe} near {unit}",
 "taint rules": "tainted input reaches {unit} ({cwe})",
 "model review":"{unit} appears to pass user input to a dangerous sink; likely {cwe}",
}
raw = [dict(f, id=f"{f['id']}/{tool}",
            message=w.format(cwe=f["cwe"], unit=f["unit"]))
       for f in findings for tool, w in sorted(ANALYSER_WORDING.items())]
print(f"raw findings from three analysers: {len(raw)}")

def dedup(rows, key):
    seen = {}
    for r in rows:
        seen.setdefault(key(r), r)
    return sorted(seen.values(), key=lambda r: r["id"])

by_identity = dedup(raw, lambda r: (r["cwe"], r["file"], r["unit"], r["evidence"]))
by_message  = dedup(raw, lambda r: r["message"])
print(f"deduped on defect identity : {len(by_identity)}")
print(f"deduped on message text    : {len(by_message)}")

bad = dict(audit, findings=by_message,
           counts=dict(audit["counts"], deduped=len(by_message),
                       verified=len(by_message),
                       feasible=sum(1 for f in by_message if f["feasible"])))
print(f"\nconformance problems: {len(check(bad, contract))}   <- still zero")
seq2 = [bad["counts"][k] for k in ("raw", "deduped", "verified", "feasible")]
print(f"counts               : {seq2}")
print()
print(f"Three defects became {len(by_message)} findings, and every one of them is")
print("schema-valid. Each analyser words the same defect differently, so the")
print("message is a unique key by construction - it deduplicates nothing while")
print("looking like it deduplicates everything.")
print()
print("The queue triples. Nobody reads the third page. The defect that gets")
print("fixed is whichever one happened to sort first.")
assert not check(bad, contract), "the broken pipeline still conforms - that is the point"
assert len(by_message) > len(by_identity), "message-keyed dedup must inflate the list"
assert len(by_identity) == len(SINKS)

## 8 · The same failure, from a real model

Everything above is constructed. Here is the identical failure produced by an actual open-weight model — **Moonlight-16B-A3B**, Moonshot AI's MoE from the Kimi team — run on a Kaggle CPU kernel against this skill's output contract.

It was given the contract and two vulnerable functions: an `open()` on a caller-supplied path, and an `os.system()` on a caller-supplied argument. Its answer is reproduced verbatim below ([full run](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/kimi/moonlight-16b-completion-prompt.txt)).

In [ ]:
# Verbatim output from Moonlight-16B-A3B on Kaggle, 2026-08-17.
# Not a paraphrase and not a stand-in: this is what the model emitted.
MODEL_OUTPUT = '''{"findings": [{"id": "F-01", "cwe": "CWE-89", "file": "report_api.py",
"line": 22, "unit": "get_report",
"evidence": "open('/var/reports/' + request.args['name'])",
"missing_control": "str", "occurrences": 1, "verdict": "confirmed",
"verdict_reason": "str", "feasible": true, "confidence": 0.0}],
"dropped": [], "counts": {"raw": 0, "deduped": 0, "verified": 0, "feasible": 0}}'''

model = json.loads(MODEL_OUTPUT)
problems = check(model, contract)
print(f"conformance problems: {len(problems)}")
print()
f = model["findings"][0]
print(f"evidence it cited : {f['evidence']}")
print(f"CWE it assigned   : {f['cwe']}  (SQL injection)")
print(f"CWE it actually is: CWE-22  (path traversal - it is open(), not a query)")
print(f"missing_control   : {f['missing_control']!r}")
print(f"verdict_reason    : {f['verdict_reason']!r}")
print(f"counts            : {model['counts']}  while findings has {len(model['findings'])}")
assert not problems, "the real model's output conforms - that is the point"

## 9 · Read that output again

It passes the contract with zero problems, and almost nothing in it is true.

In [ ]:
print("What a schema check can see:")
print(f"   every required field present, every type correct -> {len(check(model, contract))} problems")
print()
print("What it cannot see:")
print("   1. the CWE is wrong. open() on a caller-supplied path is CWE-22,")
print("      not CWE-89. The second sink, os.system(), is CWE-78 - and the")
print("      model gave that one CWE-89 as well.")
print("   2. `missing_control` and `verdict_reason` are the literal string")
print("      'str' - the model copied the contract's TYPE PLACEHOLDER into")
print("      the value. A schema saying a field must be a string is")
print("      perfectly satisfied by the word 'str'.")
print("   3. counts says 0 findings. The findings array has 1.")
print()
# monotonicity alone passes here: [0,0,0,0] is non-increasing. The invariant
# that catches this one is different, and cheap.
seq = [model["counts"][k] for k in ("raw", "deduped", "verified", "feasible")]
print(f"counts non-increasing?      {all(x >= y for x, y in zip(seq, seq[1:]))}  <- passes")
print(f"counts.verified == len(findings)?  "
      f"{model['counts']['verified'] == len(model['findings'])}  <- catches it")
print()
print("Three defects, zero schema violations. That is what a headline of")
print("'100% schema-valid' actually means as a quality metric, and it is why")
print("accuracy has to be measured against a key the model never sees.")
assert model["counts"]["verified"] != len(model["findings"])
assert f["cwe"] != "CWE-22", "the model got the weakness class wrong"
assert f["missing_control"] == "str", "the model copied the type placeholder"

## What you just proved

The call graph identifies three entry points, one of which uses dynamic dispatch. `load_report` is reachable, `debug_dump` and `legacy_export` are unknown rather than unreachable because runtime handler resolution cannot be ruled out. Two-bucket filtering silently drops both, and the three-bucket routing sends the unknowns to Phase 4 instead of paging or discarding them.

## Your turn

Count how many `unknown` cases your own reachability analysis produces, and find out what your tooling does with them. If it reports them as clean, the number of real bugs you are dropping is the size of that bucket.

---

**Next → [B1.8 · Sandbox replication](https://spbreed.github.io/cyber-commons/lessons/B1.8.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/B1.7.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/B1.7.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*